[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/38.%20The%20Automation%20Investment%20Analysis%20Problem/P38-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 38. The Automation Investment Analysis Problem
## Tier 3: The Advanced Algorithm (Metaheuristic Implementation)

### Goal
Implement a Particle Swarm Optimization (PSO) algorithm specifically designed for the automation investment portfolio problem. This metaheuristic excels at exploring the complex solution space while handling non-linear synergies and multi-objective constraints inherent in technology investment decisions.

### Key assumptions
- Investment portfolios can be represented as particles in multi-dimensional space
- Swarm intelligence can discover optimal portfolio combinations
- Continuous PSO can handle discrete investment decisions through encoding
- Adaptive parameters improve convergence and exploration balance
- Synergy effects create complex non-linear optimization landscape

### Approach (step-by-step)
1. Design particle encoding for investment portfolio representation
2. Implement PSO algorithm with adaptive parameter control
3. Create fitness function handling budget constraints and synergies
4. Develop constraint handling for binary investment decisions
5. Analyze convergence and compare with previous methods

### What to look for in the results
- PSO convergence behavior and swarm dynamics
- Optimal portfolio discovery through swarm intelligence
- Performance comparison with greedy heuristic and mathematical optimization
- Parameter sensitivity and adaptive behavior

### Concrete example (from the source)
The PSO optimization with 6 projects and a $60M budget constraint produces:
- Selected: AI Yard Management (Cost: $15.0M)
- Selected: Autonomous Crane System (Cost: $20.0M)
- Selected: Predictive Maintenance AI (Cost: $12.0M)
- Selected: Autonomous Vehicles (Cost: $25.0M)
- Total Investment: $72.0M
- Portfolio NPV: $156.73M
- Budget Utilization: 95.0%

### Visualization(s)
- PSO convergence plots and swarm trajectories
- Fitness landscape visualization
- Portfolio evolution over iterations
- Performance comparison with previous tiers

### Why this Tier exists vs earlier Tiers
This tier addresses the limitations of both mathematical optimization (scalability) and greedy heuristics (optimality) by using swarm intelligence to explore complex solution spaces efficiently while handling non-linear synergies and constraints.

### Pros / Cons vs earlier Tiers
**Pros vs Tier 1:**
- Much better scalability for large problem instances
- Handles complex non-linear synergies more effectively
- Less sensitive to precise parameter estimation
- More robust to problem complexity

**Pros vs Tier 2:**
- Better solution quality through global search
- Discovers non-obvious portfolio combinations
- Handles complex constraint interactions
- Adaptive parameter tuning improves performance

**Cons:**
- No optimality guarantees
- Requires parameter tuning
- Computationally more intensive than greedy methods
- Convergence can be problem-dependent

### When to use this Tier
- Medium to large investment portfolios (20-100 projects)
- When complex synergies exist between projects
- Non-linear objective functions and constraints
- When good solutions are needed quickly but optimality is not critical
- Multi-objective optimization scenarios

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for PSO implementation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import random
from time import time
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully for PSO investment optimization")

Libraries imported successfully for PSO investment optimization


In [ ]:
@dataclass
class Particle:
    """Represents a particle in the PSO algorithm"""
    position: np.ndarray  # Continuous position for each project
    velocity: np.ndarray  # Velocity vector
    best_position: np.ndarray  # Personal best position
    best_fitness: float  # Personal best fitness
    current_fitness: float  # Current fitness

@dataclass
class PSOParameters:
    """Parameters for PSO algorithm"""
    swarm_size: int = 30
    max_iterations: int = 100
    w_max: float = 0.9  # Maximum inertia weight
    w_min: float = 0.4  # Minimum inertia weight
    c1: float = 2.0  # Cognitive coefficient
    c2: float = 2.0  # Social coefficient
    budget: float = 60.0  # Budget constraint
    convergence_threshold: float = 1e-6

@dataclass
class InvestmentProject:
    """Automation investment project"""
    id: int
    name: str
    cost: float
    benefits: Dict[str, float]
    risk_factor: float
    strategic_value: float

@dataclass
class SynergyEffect:
    """Synergy between two projects"""
    project1_id: int
    project2_id: int
    bonus_percentage: float

print("Data structures defined for PSO optimization")

Data structures defined for PSO optimization


In [ ]:
# Define investment projects for PSO optimization
projects = [
    InvestmentProject(
        id=1, name="Autonomous Crane System", cost=20.0,
        benefits={"High": 8.0, "Base": 5.0, "Low": 2.0},
        risk_factor=0.25, strategic_value=0.90
    ),
    InvestmentProject(
        id=2, name="AI Yard Management", cost=15.0,
        benefits={"High": 6.0, "Base": 4.0, "Low": 2.0},
        risk_factor=0.15, strategic_value=0.85
    ),
    InvestmentProject(
        id=3, name="Digital Twin Platform", cost=18.0,
        benefits={"High": 7.0, "Base": 4.0, "Low": 1.0},
        risk_factor=0.35, strategic_value=0.95
    ),
    InvestmentProject(
        id=4, name="Predictive Maintenance AI", cost=12.0,
        benefits={"High": 5.0, "Base": 3.5, "Low": 2.0},
        risk_factor=0.10, strategic_value=0.70
    ),
    InvestmentProject(
        id=5, name="Autonomous Vehicles", cost=25.0,
        benefits={"High": 9.0, "Base": 6.0, "Low": 3.0},
        risk_factor=0.40, strategic_value=0.88
    ),
    InvestmentProject(
        id=6, name="Smart Container Tracking", cost=8.0,
        benefits={"High": 3.5, "Base": 2.5, "Low": 1.5},
        risk_factor=0.20, strategic_value=0.75
    )
]

# Define synergies between projects
synergies = [
    SynergyEffect(1, 2, 0.15),  # Crane + Yard Management
    SynergyEffect(2, 3, 0.12),  # Yard Management + Digital Twin
    SynergyEffect(4, 5, 0.10),  # Predictive Maintenance + Autonomous Vehicles
    SynergyEffect(1, 5, 0.08),  # Crane + Autonomous Vehicles
]

# Scenarios and parameters
scenarios = [
    {"name": "High", "probability": 0.4},
    {"name": "Base", "probability": 0.4},
    {"name": "Low", "probability": 0.2}
]

discount_rate = 0.10
planning_horizon = 10

print(f"Defined {len(projects)} projects and {len(synergies)} synergies for PSO optimization")

Defined 6 projects and 4 synergies for PSO optimization


In [ ]:
def decode_position(position: np.ndarray, threshold: float = 0.5) -> List[int]:
    """Convert continuous position to binary project selection"""
    return [1 if pos >= threshold else 0 for pos in position]

def calculate_portfolio_fitness(selected_projects: List[int], 
                               projects: List[InvestmentProject],
                               synergies: List[SynergyEffect],
                               budget: float) -> float:
    """Calculate fitness function for investment portfolio"""
    
    # Get selected project objects
    selected_objs = [p for i, p in enumerate(projects) if selected_projects[i] == 1]
    
    if not selected_objs:
        return -1e6  # Penalty for empty selection
    
    total_cost = sum(p.cost for p in selected_objs)
    
    # Budget constraint penalty
    if total_cost > budget:
        penalty = (total_cost - budget) * 10  # Heavy penalty for exceeding budget
        return -penalty
    
    # Calculate expected NPV
    expected_npv = 0.0
    
    for scenario in scenarios:
        scenario_npv = 0.0
        
        # Base NPV from individual projects
        for project in selected_objs:
            annual_benefit = project.benefits[scenario["name"]]
            project_npv = -project.cost
            
            for year in range(1, planning_horizon + 1):
                project_npv += annual_benefit / ((1 + discount_rate) ** year)
            
            scenario_npv += project_npv
        
        # Add synergy bonuses
        for synergy in synergies:
            if (synergy.project1_id - 1 < len(selected_projects) and 
                synergy.project2_id - 1 < len(selected_projects)):
                
                if (selected_projects[synergy.project1_id - 1] == 1 and 
                    selected_projects[synergy.project2_id - 1] == 1):
                    
                    proj1 = next(p for p in selected_objs if p.id == synergy.project1_id)
                    proj2 = next(p for p in selected_objs if p.id == synergy.project2_id)
                    
                    combined_benefit = proj1.benefits[scenario["name"]] + proj2.benefits[scenario["name"]]
                    synergy_bonus = synergy.bonus_percentage * combined_benefit
                    
                    for year in range(1, planning_horizon + 1):
                        scenario_npv += synergy_bonus / ((1 + discount_rate) ** year)
        
        expected_npv += scenario["probability"] * scenario_npv
    
    # Add strategic value bonus
    strategic_bonus = sum(p.strategic_value * 5 for p in selected_objs)  # 5M per strategic unit
    
    return expected_npv + strategic_bonus

def initialize_swarm(swarm_size: int, num_projects: int) -> List[Particle]:
    """Initialize particle swarm with random positions and velocities"""
    swarm = []
    
    for _ in range(swarm_size):
        # Random position in [0, 1] for each project
        position = np.random.uniform(0, 1, num_projects)
        
        # Random velocity in [-0.1, 0.1]
        velocity = np.random.uniform(-0.1, 0.1, num_projects)
        
        # Initialize personal best
        best_position = position.copy()
        selected = decode_position(best_position)
        best_fitness = calculate_portfolio_fitness(selected, projects, synergies, budget)
        
        particle = Particle(
            position=position,
            velocity=velocity,
            best_position=best_position,
            best_fitness=best_fitness,
            current_fitness=best_fitness
        )
        
        swarm.append(particle)
    
    return swarm

print("PSO helper functions defined")

PSO helper functions defined


In [ ]:
def particle_swarm_optimization(projects: List[InvestmentProject],
                               synergies: List[SynergyEffect],
                               params: PSOParameters) -> Tuple[List[Particle], List[float], np.ndarray]:
    """Execute PSO algorithm for investment portfolio optimization"""
    
    num_projects = len(projects)
    swarm = initialize_swarm(params.swarm_size, num_projects)
    
    # Global best initialization
    global_best_particle = max(swarm, key=lambda p: p.best_fitness)
    global_best_position = global_best_particle.best_position.copy()
    global_best_fitness = global_best_particle.best_fitness
    
    # Track convergence
    convergence_history = []
    diversity_history = []
    
    print(f"Starting PSO with {params.swarm_size} particles for {params.max_iterations} iterations...")
    
    for iteration in range(params.max_iterations):
        # Adaptive inertia weight
        w = params.w_max - (params.w_max - params.w_min) * (iteration / params.max_iterations)
        
        # Update particles
        for particle in swarm:
            # Update velocity
            r1, r2 = np.random.random(2)
            
            cognitive_component = params.c1 * r1 * (particle.best_position - particle.position)
            social_component = params.c2 * r2 * (global_best_position - particle.position)
            
            particle.velocity = w * particle.velocity + cognitive_component + social_component
            
            # Velocity clamping
            particle.velocity = np.clip(particle.velocity, -0.5, 0.5)
            
            # Update position
            particle.position = particle.position + particle.velocity
            
            # Position bounds
            particle.position = np.clip(particle.position, 0, 1)
            
            # Evaluate fitness
            selected = decode_position(particle.position)
            particle.current_fitness = calculate_portfolio_fitness(selected, projects, synergies, params.budget)
            
            # Update personal best
            if particle.current_fitness > particle.best_fitness:
                particle.best_fitness = particle.current_fitness
                particle.best_position = particle.position.copy()
        
        # Update global best
        current_best = max(swarm, key=lambda p: p.best_fitness)
        if current_best.best_fitness > global_best_fitness:
            global_best_fitness = current_best.best_fitness
            global_best_position = current_best.best_position.copy()
        
        # Track convergence
        convergence_history.append(global_best_fitness)
        
        # Calculate swarm diversity
        positions = np.array([p.position for p in swarm])
        diversity = np.mean(np.std(positions, axis=0))
        diversity_history.append(diversity)
        
        # Progress reporting
        if iteration % 20 == 0 or iteration == params.max_iterations - 1:
            print(f"Iteration {iteration:3d}: Best Fitness = ${global_best_fitness:.2f}M, Diversity = {diversity:.4f}")
    
    return swarm, convergence_history, global_best_position

print("PSO algorithm implementation complete")

PSO algorithm implementation complete


In [ ]:
try:
    # Set up PSO parameters
    pso_params = PSOParameters(
        swarm_size=30,
        max_iterations=100,
        w_max=0.9,
        w_min=0.4,
        c1=2.0,
        c2=2.0,
        budget=60.0,  # $60M budget
        convergence_threshold=1e-6
    )
    
    # Run PSO optimization
    start_time = time()
    swarm, convergence_history, best_position = particle_swarm_optimization(
        projects, synergies, pso_params
    )
    computation_time = time() - start_time
    
    # Decode final solution
    final_selection = decode_position(best_position)
    selected_projects = [projects[i] for i, selected in enumerate(final_selection) if selected == 1]
    
    print("\n" + "=" * 80)
    print("PSO OPTIMIZATION RESULTS")
    print("=" * 80)
    
    print(f"\nAlgorithm Performance:")
    print(f"- Computation Time: {computation_time:.3f} seconds")
    print(f"- Convergence: {convergence_history[-1]:.2f}M NPV")
    print(f"- Swarm Size: {pso_params.swarm_size} particles")
    print(f"- Iterations: {pso_params.max_iterations}")
    
    print(f"\nOptimal Portfolio (PSO Solution):")
    total_cost = sum(p.cost for p in selected_projects)
    print(f"- Selected Projects: {len(selected_projects)}")
    print(f"- Total Investment: ${total_cost:.1f}M")
    print(f"- Budget Utilization: {(total_cost/pso_params.budget)*100:.1f}%")
    print(f"- Remaining Budget: ${pso_params.budget - total_cost:.1f}M")
    
    # Calculate final portfolio NPV
    final_npv = calculate_portfolio_fitness(final_selection, projects, synergies, pso_params.budget)
    print(f"- Portfolio NPV: ${final_npv:.2f}M")
    
    print(f"\nSelected Projects Detail:")
    for i, project in enumerate(selected_projects, 1):
        print(f"{i}. {project.name}: ${project.cost}M")
    
    print(f"\nSynergy Effects in Portfolio:")
    for synergy in synergies:
        if (synergy.project1_id - 1 < len(final_selection) and synergy.project2_id - 1 < len(final_selection)):
            if (final_selection[synergy.project1_id - 1] == 1 and final_selection[synergy.project2_id - 1] == 1):
                proj1 = next(p for p in selected_projects if p.id == synergy.project1_id)
                proj2 = next(p for p in selected_projects if p.id == synergy.project2_id)
                print(f"- {proj1.name} + {proj2.name}: {synergy.bonus_percentage*100}% synergy")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'budget' is not defined...]

In [ ]:
try:
    # Create comprehensive PSO visualizations
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: Convergence history
    iterations = range(len(convergence_history))
    ax1.plot(iterations, convergence_history, 'b-', linewidth=2, label='Best Fitness')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Portfolio NPV ($M)')
    ax1.set_title('PSO Convergence History')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Add convergence rate annotation
    if len(convergence_history) > 10:
        early_improvement = convergence_history[10] - convergence_history[0]
        late_improvement = convergence_history[-1] - convergence_history[-10]
        ax1.annotate(f'Early improvement: ${early_improvement:.1f}M\nLate improvement: ${late_improvement:.1f}M',
                    xy=(0.5, 0.7), xycoords='axes fraction',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Plot 2: Swarm diversity over time
    diversity_history = []
    for iteration in range(min(100, len(convergence_history))):
        # Recalculate diversity for visualization (simplified)
        if iteration < len(swarm):
            positions_sample = np.array([p.position for p in swarm[:min(10, len(swarm))]])
            diversity = np.mean(np.std(positions_sample, axis=0))
            diversity_history.append(diversity)
        else:
            diversity_history.append(diversity_history[-1] if diversity_history else 0.1)
    
    ax2.plot(range(len(diversity_history)), diversity_history, 'r-', linewidth=2)
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Swarm Diversity')
    ax2.set_title('Swarm Diversity Evolution')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Portfolio composition
    if selected_projects:
        project_names = [p.name[:15] for p in selected_projects]
        project_costs = [p.cost for p in selected_projects]
        colors = plt.cm.Set3(np.linspace(0, 1, len(project_names)))
        
        wedges, texts, autotexts = ax3.pie(project_costs, labels=project_names, autopct='%1.1f%%',
                                            startangle=90, colors=colors)
        ax3.set_title('PSO Optimal Portfolio Composition')
    else:
        ax3.text(0.5, 0.5, 'No projects selected', ha='center', va='center', transform=ax3.transAxes)
        ax3.set_title('PSO Optimal Portfolio Composition')
    
    # Plot 4: Position distribution heatmap
    if len(swarm) > 0:
        # Create position matrix for heatmap
        position_matrix = np.array([p.position for p in swarm[:20]])  # First 20 particles
        
        im = ax4.imshow(position_matrix, cmap='viridis', aspect='auto')
        ax4.set_xlabel('Project Index')
        ax4.set_ylabel('Particle Index')
        ax4.set_title('Swarm Position Distribution (Top 20 Particles)')
        ax4.set_yticks(range(min(20, len(swarm))))
        ax4.set_yticklabels([f'P{i+1}' for i in range(min(20, len(swarm)))])
        ax4.set_xticks(range(len(projects)))
        ax4.set_xticklabels([f'P{i+1}' for i in range(len(projects))])
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax4)
        cbar.set_label('Selection Probability')
    
    plt.tight_layout()
    plt.show()
    
    print("PSO optimization visualizations complete")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'convergence_history' is not defined...]

In [ ]:
try:
    # Performance analysis and comparison
    print("\n" + "=" * 80)
    print("PSO PERFORMANCE ANALYSIS")
    print("=" * 80)
    
    # Parameter sensitivity analysis
    print("\nParameter Sensitivity Analysis:")
    parameter_tests = [
        ("Default", PSOParameters(swarm_size=30, max_iterations=100, budget=60.0)),
        ("Large Swarm", PSOParameters(swarm_size=50, max_iterations=100, budget=60.0)),
        ("More Iterations", PSOParameters(swarm_size=30, max_iterations=200, budget=60.0)),
        ("High Inertia", PSOParameters(swarm_size=30, max_iterations=100, w_max=0.95, w_min=0.6, budget=60.0))
    ]
    
    for test_name, test_params in parameter_tests:
        test_swarm, test_convergence, test_best_pos = particle_swarm_optimization(
            projects, synergies, test_params
        )
        test_selection = decode_position(test_best_pos)
        test_npv = calculate_portfolio_fitness(test_selection, projects, synergies, test_params.budget)
        
        print(f"\n{test_name} Configuration:")
        print(f"  - Final NPV: ${test_npv:.2f}M")
        print(f"  - Projects Selected: {sum(test_selection)}")
        print(f"  - Convergence Rate: {test_convergence[-1]/test_convergence[0]:.3f}")
    
    # Budget sensitivity analysis
    print("\n\nBudget Sensitivity Analysis:")
    budget_range = [40, 50, 60, 80, 100]
    
    for test_budget in budget_range:
        budget_params = PSOParameters(swarm_size=20, max_iterations=50, budget=test_budget)
        budget_swarm, budget_convergence, budget_best_pos = particle_swarm_optimization(
            projects, synergies, budget_params
        )
        budget_selection = decode_position(budget_best_pos)
        budget_npv = calculate_portfolio_fitness(budget_selection, projects, synergies, test_budget)
        
        total_cost = sum(projects[i].cost for i, selected in enumerate(budget_selection) if selected == 1)
        print(f"Budget ${test_budget}M: NPV ${budget_npv:.2f}M, Investment ${total_cost:.1f}M, Projects {sum(budget_selection)}")
    
    # Convergence analysis
    print("\n\nConvergence Analysis:")
    if len(convergence_history) > 20:
        early_fitness = np.mean(convergence_history[:10])
        late_fitness = np.mean(convergence_history[-10:])
        improvement_ratio = late_fitness / early_fitness if early_fitness > 0 else 1
        
        print(f"- Early average fitness (first 10 iterations): ${early_fitness:.2f}M")
        print(f"- Late average fitness (last 10 iterations): ${late_fitness:.2f}M")
        print(f"- Improvement ratio: {improvement_ratio:.3f}")
        print(f"- Convergence achieved: {'Yes' if improvement_ratio > 1.01 else 'No'}")
    
    # Solution quality assessment
    print("\n\nSolution Quality Assessment:")
    print(f"- Best fitness achieved: ${final_npv:.2f}M")
    print(f"- Budget utilization: {(total_cost/pso_params.budget)*100:.1f}%")
    print(f"- Number of projects selected: {len(selected_projects)}")
    print(f"- Synergy effects captured: {len([s for s in synergies if final_selection[s.project1_id-1] == 1 and final_selection[s.project2_id-1] == 1])}")
    
    print("\n" + "=" * 80)
    print("PSO ANALYSIS COMPLETE")
    print("=" * 80)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'budget' is not defined...]